# Validation Test Evaluation (Omega study) - Capillary Rise (benchmark test case within SFB 1194)

This notebook and the corresponding startUp and simulation setup notebook (CapillaryRise_SFB1194_Run.ipynb, CapillaryRise_SFB1194_RunStartUp.ipynb) are part of published results (cf. 4.6) found in *A comparative study of transient capillary rise using direct numerical simulations* (https://doi.org/10.1016/j.apm.2020.04.020). 

In [ ]:
#r "BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Release/net8.0/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
BoSSSshell.WorkflowMgm.Init("CapillaryRisePaper");

Opening existing database 'C:\BoSSS-localJobs\CapillaryRisePaper_OnJenkins'.


In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

In [4]:
bool runShortTests = true; // set to 'false' to run all test cases for the whole period of time (up to 2.7), otherwise they will run only up to 0.4

## Sessions

In [ ]:
static var sessions = BoSSSshell.WorkflowMgm.Sessions.ToList();
//static var sessions = databases.Pick(0).Sessions;
sessions

#0: CapillaryRisePaper	CapillaryRise_Omega1_resolvedSlipR5_OmegaStudy_mesh8	12/10/2025 5:26:35 PM	b4015311...
#1: CapillaryRisePaper	CapillaryRise_Omega10_resolvedSlipR5_OmegaStudy_mesh8	12/10/2025 5:30:15 PM	65f65dcb...
#2: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh1_startUp	11/24/2025 3:07:56 PM	243a4381...
#3: CapillaryRisePaper	CapillaryRise_Omega100_OmegaStudy_mesh8_startUp	11/24/2025 3:09:20 PM	e34da697...
#4: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh2_startUp	11/24/2025 3:08:05 PM	ca56d38d...
#5: CapillaryRisePaper	CapillaryRise_Omega10_OmegaStudy_mesh8_startUp	11/24/2025 3:09:09 PM	6c1e7c74...
#6: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh4_startUp	11/24/2025 3:08:13 PM	0d08da88...
#7: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh8_startUp*	11/24/2025 3:08:23 PM	52e81583...
#8: CapillaryRisePaper	CapillaryRise_Omega05_OmegaStudy_mesh8_AMR1_startUp*	11/24/2025 3:08:51 PM	89838821...
#9: CapillaryRisePaper	CapillaryRise_Omega01_OmegaS

## Omega study

In [6]:
string[] OmegaS = { "Omega01", "Omega05", "Omega1", "Omega10", "Omega100" };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Nov26_000000.CapillaryRisePaper");

Opening existing database 'C:\BoSSS-localJobs\bkup-2025Nov26_000000.CapillaryRisePaper'.


In [8]:
List<ISessionInfo> evalSessOrigin = new List<ISessionInfo>();

In [9]:
foreach (string Omega in OmegaS) {

    string studyName = $"CapillaryRise_{Omega}_resolvedSlipR5_OmegaStudy_mesh8";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));

    int studyCount = studySess.Count();
    if (studyCount == 0) {
        Console.WriteLine("Not found in orignDB!");
    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSessOrigin.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSessOrigin.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
evalSessOrigin

Searching for: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8
Restart session found! Take last run
Searching for: CapillaryRise_Omega05_resolvedSlipR5_OmegaStudy_mesh8
found
Searching for: CapillaryRise_Omega1_resolvedSlipR5_OmegaStudy_mesh8
found
Searching for: CapillaryRise_Omega10_resolvedSlipR5_OmegaStudy_mesh8
found
Searching for: CapillaryRise_Omega100_resolvedSlipR5_OmegaStudy_mesh8
found


#0: CapillaryRisePaper	CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart7*	5/5/2019 8:39:07 PM	ac7ca537...
#1: CapillaryRisePaper	CapillaryRise_Omega05_resolvedSlipR5_OmegaStudy_mesh8	10/10/2018 10:26:14 AM	c641081c...
#2: CapillaryRisePaper	CapillaryRise_Omega1_resolvedSlipR5_OmegaStudy_mesh8	10/10/2018 10:28:48 AM	407e1ada...
#3: CapillaryRisePaper	CapillaryRise_Omega10_resolvedSlipR5_OmegaStudy_mesh8	10/10/2018 10:34:57 AM	e522839b...
#4: CapillaryRisePaper	CapillaryRise_Omega100_resolvedSlipR5_OmegaStudy_mesh8	10/10/2018 10:40:22 AM	0fd54aac...


In [10]:
NUnit.Framework.Assert.AreEqual(5, evalSessOrigin.Count, $"Not enough sessions for evaluation.");

In [11]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [12]:
foreach (string Omega in OmegaS) {

    string studyName = $"CapillaryRise_{Omega}_resolvedSlipR5_OmegaStudy_mesh8";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));

    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found in wmg!");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }
    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }

}
evalSess

Searching for: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8
Restart session found in originDB! Take last run
Searching for: CapillaryRise_Omega05_resolvedSlipR5_OmegaStudy_mesh8
found in originDB
Searching for: CapillaryRise_Omega1_resolvedSlipR5_OmegaStudy_mesh8
found
Searching for: CapillaryRise_Omega10_resolvedSlipR5_OmegaStudy_mesh8
found
Searching for: CapillaryRise_Omega100_resolvedSlipR5_OmegaStudy_mesh8
found in originDB


#0: CapillaryRisePaper	CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart7*	5/5/2019 8:39:07 PM	ac7ca537...
#1: CapillaryRisePaper	CapillaryRise_Omega05_resolvedSlipR5_OmegaStudy_mesh8	10/10/2018 10:26:14 AM	c641081c...
#2: CapillaryRisePaper	CapillaryRise_Omega1_resolvedSlipR5_OmegaStudy_mesh8	12/10/2025 5:26:35 PM	b4015311...
#3: CapillaryRisePaper	CapillaryRise_Omega10_resolvedSlipR5_OmegaStudy_mesh8	12/10/2025 5:30:15 PM	65f65dcb...
#4: CapillaryRisePaper	CapillaryRise_Omega100_resolvedSlipR5_OmegaStudy_mesh8	10/10/2018 10:40:22 AM	0fd54aac...


In [13]:
NUnit.Framework.Assert.AreEqual(5, evalSess.Count, $"Not enough sessions for evaluation.");

In [14]:
sessions.AddRange(originDB.Sessions);
sessions

#0: CapillaryRisePaper	CapillaryRise_Omega1_resolvedSlipR5_OmegaStudy_mesh8	12/10/2025 5:26:35 PM	b4015311...
#1: CapillaryRisePaper	CapillaryRise_Omega10_resolvedSlipR5_OmegaStudy_mesh8	12/10/2025 5:30:15 PM	65f65dcb...
#2: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh1_startUp	11/24/2025 3:07:56 PM	243a4381...
#3: CapillaryRisePaper	CapillaryRise_Omega100_OmegaStudy_mesh8_startUp	11/24/2025 3:09:20 PM	e34da697...
#4: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh2_startUp	11/24/2025 3:08:05 PM	ca56d38d...
#5: CapillaryRisePaper	CapillaryRise_Omega10_OmegaStudy_mesh8_startUp	11/24/2025 3:09:09 PM	6c1e7c74...
#6: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh4_startUp	11/24/2025 3:08:13 PM	0d08da88...
#7: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh8_startUp*	11/24/2025 3:08:23 PM	52e81583...
#8: CapillaryRisePaper	CapillaryRise_Omega05_OmegaStudy_mesh8_AMR1_startUp*	11/24/2025 3:08:51 PM	89838821...
#9: CapillaryRisePaper	CapillaryRise_Omega01_OmegaS

### compute times

In [15]:
foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    List<string> dtS = new List<string>();
    dtS.Add(currSI.KeysAndQueries["dtFixed"].ToString());
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        //Console.WriteLine($"   processing session {currSI.Name}");
        var rSI = rSIs.Single();
        if (rSI.Name.ToLower().Contains("_startup"))
            break;
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        dtS.Add(currSI.KeysAndQueries["dtFixed"].ToString());
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    timesteps -= currTimesteps.First().TimeStepNumber.MajorNumber;
    physicalTime -= currTimesteps.First().PhysicalTime;

    //Console.WriteLine($"Session - {sessName}: timesteps = {timesteps}, t_end = {physicalTime} (saveperiod = {currSI.KeysAndQueries["saveperiod"]}, LogPeriod = {currSI.KeysAndQueries["LogPeriod"]}); compute time = {computeTime.Days}:{computeTime.Hours}");
    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps}, t_end = {physicalTime}; compute time = {computeTime.Days}:{computeTime.Hours}");
    
    for (int i = 0; i < dtS.Count; i++) {
        Console.WriteLine($"  dt sess {i}: {dtS[i]}");
    }
}

Session - CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1: timesteps = 282600, t_end = 13.628000000016938; compute time = 54:23
  dt sess 0: 2.5E-05
  dt sess 1: 5E-05
  dt sess 2: 2.5E-05
  dt sess 3: 5E-05
  dt sess 4: 0.0001
  dt sess 5: 2E-05
  dt sess 6: 2E-05
  dt sess 7: 2E-05
Session - CapillaryRise_Omega05_resolvedSlipR5_OmegaStudy_mesh8: timesteps = 11100, t_end = 1.109999999999878; compute time = 1:15
  dt sess 0: 0.0001
Session - CapillaryRise_Omega1_resolvedSlipR5_OmegaStudy_mesh8: timesteps = 4000, t_end = 0.39999999999995595; compute time = 0:4
  dt sess 0: 0.0001
Session - CapillaryRise_Omega10_resolvedSlipR5_OmegaStudy_mesh8: timesteps = 4000, t_end = 0.3999999999999622; compute time = 0:5
  dt sess 0: 0.0001
Session - CapillaryRise_Omega100_resolvedSlipR5_OmegaStudy_mesh8: timesteps = 55426, t_end = 27.712999999979832; compute time = 8:11
  dt sess 0: 0.0005


### read log data

In [16]:
public static List<Plot2Ddata>[] ReadMergedDataForMovingContactLine(this IEnumerable<ISessionInfo> sess) {

    List<Plot2Ddata>[] evalData = new List<Plot2Ddata>[2];

    foreach (var evalSess in sess) {
        Console.WriteLine("Processing session: " + evalSess.Name);
        // Get session history
        List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
        restartSessionList.Add(evalSess);
        Guid restartID = evalSess.RestartedFrom;
        while(restartID != Guid.Empty) {
            try {
                var rSess = sessions.Where(sess => sess.ID == restartID).Single();
                if (rSess.Name.ToLower().Contains("_startup"))
                    break;
                Console.WriteLine("  Found restart session: " + rSess.Name);
                restartSessionList.Add(rSess);
                restartID = rSess.RestartedFrom;
            }
            catch { // do nothing if session not found
                restartID = Guid.Empty;
            } 
        }
        restartSessionList.Reverse();

        Console.WriteLine("  Merging data from " + restartSessionList.Count + " sessions.");
        // Read data from all sessions and merge
        var mergedEvalData = restartSessionList.ReadLogDataForMovingContactLine(mergeSess: true);

        for (int i = 0; i < 2; i++) {
            if (evalData[i] == null) {
                evalData[i] = mergedEvalData[i];
            } else {
                for (int j = 0; j < evalData[i].Count(); j++) {
                    evalData[i].ElementAt(j).AddDataGroup(mergedEvalData[i].ElementAt(j).dataGroups.First()); // add first data group (should be only one if mergeSess = true)
                }
            }
        }
    }

    return evalData;
}

In [17]:
var evalData = evalSess.ReadMergedDataForMovingContactLine();

Processing session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart7
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart6
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart5
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart4
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart3
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart2
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart1
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1
  Merging data from 8 sessions.
number of contact lines: 2
Element at 0: time vs contact-pointX
Element at 1: time vs contact-pointY
Element at 2: time vs contact-VelocityX
Element at 3: time vs contact-VelocityY
Element at 4: time vs contact-angle
Processing session: C

In [18]:
var evalDataOrigin = evalSessOrigin.ReadMergedDataForMovingContactLine();

Processing session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart7
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart6
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart5
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart4
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart3
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart2
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1_restart1
  Found restart session: CapillaryRise_Omega01_resolvedSlipR5_OmegaStudy_mesh8_AMR1
  Merging data from 8 sessions.
number of contact lines: 2
Element at 0: time vs contact-pointX
Element at 1: time vs contact-pointY
Element at 2: time vs contact-VelocityX
Element at 3: time vs contact-VelocityY
Element at 4: time vs contact-angle
Processing session: C

In [19]:
public static List<Plot2Ddata>[] CleanUpDataForMovingContactLine(this List<Plot2Ddata>[] data) {

    List<Plot2Ddata>[] evalData = new List<Plot2Ddata>[2];
    evalData[0] = new List<Plot2Ddata>();   // contact line at symmetry plane
    evalData[1] = new List<Plot2Ddata>();   // contact line at wall 

    int numDataGroups = data[0].ElementAt(0).dataGroups.Count();
    for (int grp = 0; grp < numDataGroups; grp++) {

        //Console.WriteLine("Processing data group: " + data[0].ElementAt(0).dataGroups[grp].Name);

        // get index permutation to sort data according to time
        double[] cLpos = data[0].ElementAt(0).dataGroups[grp].Values;   // check the X-position of the contact line
        int numTimesteps = cLpos.Length;
        int[][] cLIndex = new int[2][];
        cLIndex[0] = new int[numTimesteps];   // contact line at symmetry plane
        cLIndex[1] = new int[numTimesteps];   // contact line at wall
        for (int ts = 0; ts < numTimesteps; ts++) {
            if (cLpos[ts] < 0.0025) {   // contact line at symmetry plane
                cLIndex[0][ts] = 0;
                cLIndex[1][ts] = 1;
            } else {    // contact line at wall
                cLIndex[0][ts] = 1;
                cLIndex[1][ts] = 0;
            }
        }

        // get t0
        double t0 = data[0].ElementAt(0).dataGroups[grp].Abscissas[0];

        // create new data sets
        for (int cL = 0; cL < 2; cL++) {   // cL = 0: contact line at symmetry plane; cL = 1: contact line at wall
            int numData = data[0].Count();
            for (int i = 0; i < numData; i++) {

                //Console.WriteLine($"...  Processing data set at index {i} ...");

                double[] time = new double[numTimesteps];
                double[] value = new double[numTimesteps];

                for (int ts = 0; ts < numTimesteps; ts++) {
                    time[ts] = data[0].ElementAt(i).dataGroups[grp].Abscissas[ts] - t0;
                    value[ts] = data[cLIndex[cL][ts]].ElementAt(i).dataGroups[grp].Values[ts];
                }

                if (evalData[cL].Count() <= i) {
                    Plot2Ddata plotData = new Plot2Ddata();
                    evalData[cL].Add(plotData);
                }   
                evalData[cL].ElementAt(i).AddDataGroup(data[cL].ElementAt(i).dataGroups[grp].Name, time, value);     

                //Console.WriteLine($"... done"); 
            }
        }

    }
    
    return evalData;
}

In [20]:
var cleanEvalData = evalData.CleanUpDataForMovingContactLine();

In [21]:
var cleanEvalDataOrigin = evalDataOrigin.CleanUpDataForMovingContactLine();

### reference rise height 

In [22]:
static double R = 0.005 * 1000.0; // in mm
static double Theta = 30.0 * Math.PI / 180.0; // in rad
static double riseHeight = R * (4.0 - (2.0 - Math.Sin(Theta) - (Math.Asin(Math.Cos(Theta))/Math.Cos(Theta)))/(2.0 * Math.Cos(Theta))); // in mm

riseHeight

19.160531485066464

In [23]:
Plot2Ddata refPlt = new Plot2Ddata();
refPlt.AddDataGroup("rise height", new double[]{ 0.0, 0.7 }, new double[] { riseHeight, riseHeight }, 
new PlotFormat() { Style = Styles.Lines, LineWidth = 2, LineColor = LineColors.Black });

### Fig. 15 - Numerical solutions for the Omega-study

In [24]:
List<Plot2Ddata>[] plotEvalData = (runShortTests) ? cleanEvalDataOrigin : cleanEvalData;

In [25]:
public static Plot2Ddata GetCapillaryRiseHeight_Plot2Ddata(List<Plot2Ddata>[] evalData, string omegaStudy, double tend) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "t in s";
    plt.Ylabel = "h in mm";

    var fmt = new PlotFormat();
    fmt.Style = Styles.Lines;
    fmt.LineWidth = 2;
    fmt.LineColor = (LineColors)(0);

    var datGrp = evalData[0].ElementAt(1).dataGroups.Where(grp => grp.Name.Contains(omegaStudy)).Single();
    double[] values_mm = datGrp.Values.Select(v => v * 1000.0).ToArray(); // convert from m to mm
    plt.AddDataGroup(omegaStudy.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries)[0], datGrp.Abscissas, values_mm, fmt);

    plt.XrangeMin = 0.0;
    plt.XrangeMax = tend;

    plt.ShowLegend = true;

    return plt;        
}

In [26]:
Plot2Ddata[,] Fig15 = new Plot2Ddata[2, 2];

// double R = 0.005 * 1000.0; // in mm
// double Theta = 30.0 * Math.PI / 180.0; // in rad
// double riseHeight = R * (4.0 - (2.0 - Math.Sin(Theta) - (Math.Asin(Math.Cos(Theta))/Math.Cos(Theta)))/(2.0 * Math.Cos(Theta))); // in mm

var plt = GetCapillaryRiseHeight_Plot2Ddata(plotEvalData, "_Omega01_", 13.86);
plt.AddDataGroup("rise height", new double[]{ 0.0, plt.dataGroups[0].Abscissas.Max() }, new double[] { riseHeight, riseHeight }, 
                    new PlotFormat() { Style = Styles.Lines, LineWidth = 2, LineColor = LineColors.Black });
plt.YrangeMin = 8.0;
plt.YrangeMax = 33.0;
Fig15[0, 0] = plt;

plt = GetCapillaryRiseHeight_Plot2Ddata(plotEvalData, "_Omega05_", 1.11);
plt.AddDataGroup("rise height", new double[]{ 0.0, plt.dataGroups[0].Abscissas.Max() }, new double[] { riseHeight, riseHeight }, 
                    new PlotFormat() { Style = Styles.Lines, LineWidth = 2, LineColor = LineColors.Black });
plt.YrangeMin = 8.0;
plt.YrangeMax = 29.0;
Fig15[0, 1] = plt;

plt = GetCapillaryRiseHeight_Plot2Ddata(plotEvalData, "_Omega1_", 0.69);
plt.AddDataGroup("rise height", new double[]{ 0.0, plt.dataGroups[0].Abscissas.Max() }, new double[] { riseHeight, riseHeight }, 
                    new PlotFormat() { Style = Styles.Lines, LineWidth = 2, LineColor = LineColors.Black });
plt.YrangeMin = 8.0;
plt.YrangeMax = 25.0;
Fig15[1, 0] = plt;

plt = GetCapillaryRiseHeight_Plot2Ddata(plotEvalData, "_Omega10_", 27.713);
plt.AddDataGroup(GetCapillaryRiseHeight_Plot2Ddata(plotEvalData, "_Omega100_", 27.713).dataGroups[0]);
plt.AddDataGroup("rise height", new double[]{ 0.0, plt.dataGroups[1].Abscissas.Max() }, new double[] { riseHeight, riseHeight }, 
                    new PlotFormat() { Style = Styles.Lines, LineWidth = 2, LineColor = LineColors.Black });
plt.YrangeMin = 8.0;
plt.YrangeMax = 25.0;
plt.LegendAlignment = new string[] { "i", "b", "r" };
Fig15[1, 1] = plt;

Fig15.ToGnuplot().PlotSVG(xRes:1200,yRes:1000)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 15 
 
 
 
 
 20 
 
 
 
 
 25 
 
 
 
 
 30 
 
 
 
 
 0 
 
 
 
 
 2 
 
 
 
 
 4 
 
 
 
 
 6 
 
 
 
 
 8 
 
 
 
 
 10 
 
 
 
 
 12 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 h in mm 
 
 
 
 
 t in s 
 
 
 
 
 Omega01 
 
 
 
 
 Omega01 
 
 
 
	<path stroke='gray' d='M349.2,62.1 L402.6,62.1 M69.5,418.2 L69.6,418.1 L69.6,418.0 L69.7,417.9 L69.8,417.7 L69.8,417.3
 L69.9,416.9 L70.0,416.5 L70.0,416.0 L70.1,415.4 L70.2,414.8 L70.3,414.2 L70.3,413.4 L70.4,412.7
 L70.5,411.8 L70.5,410.9 L70.6,409.9 L70.7,408.9 L70.7,407.8 L70.8,406.7 L70.9,405.5 L70.9,404.3
 L71.0,403.0 L71.1,401.7 L71.1,400.3 L71.2,398.8 L71.3,397.3 L71.3,395.8 L71.4,394.2 L71.5,392.6
 L71.6,390.9 L71.6,389.2 L71.7,387.4 L71.8,385.6 L71.8,383.8 L71.9,381.8 L72.0,380.0 L72.0,378.1
 L72.1,376.0 L72.2,374.1 L72.2,372.0 L72.3,369.9 L72.4,367.9 L72.4,365.7 L72.5,363.5 L72.6,361.3
 L72.7,359.1 L72.7,356.9 L72.8,354.6 L72.9,352.4 L72.9,350.0 L73.0,347.7 L73.1,345.3 L73.1,343.0
 L73.2,340.5 L73.3,338.2 L73.3,335.7 L73.4,333.2 L73.5,330.8 L73.5,328.3 L73.6,325.8 L73.7,323.3
 L73.7,320.8 L73.8,318.3 L73.9,315.7 L74.0,313.2 L74.0,310.6 L74.1,308.1 L74.2,305.5 L74.2,303.0
 L74.3,300.4 L74.4,297.8 L74.4,295.2 L74.5,292.6 L74.6,290.0 L74.6,287.3 L74.7,284.7 L74.8,282.0
 L74.8,279.3 L74.9,276.7 L75.0,274.0 L75.0,271.4 L75.1,268.7 L75.2,266.1 L75.3,263.4 L75.3,260.8
 L75.4,258.2 L75.5,255.6 L75.5,253.0 L75.6,250.4 L75.7,247.8 L75.7,245.2 L75.8,242.7 L75.9,240.1
 L75.9,237.6 L76.0,235.1 L76.1,232.5 L76.1,230.0 L76.2,227.4 L76.3,224.9 L76.3,222.4 L76.4,219.9
 L76.5,217.3 L76.6,214.8 L76.6,212.3 L76.7,209.8 L76.8,207.4 L76.8,204.9 L76.9,202.5 L77.0,200.0
 L77.0,197.6 L77.1,195.2 L77.2,192.9 L77.2,190.5 L77.3,188.2 L77.4,185.9 L77.4,183.6 L77.5,181.3
 L77.6,179.1 L77.6,176.8 L77.7,174.6 L77.8,172.4 L77.9,170.2 L77.9,168.0 L78.0,165.8 L78.1,163.6
 L78.1,161.5 L78.2,159.3 L78.3,157.2 L78.3,155.1 L78.4,153.0 L78.5,150.9 L78.5,148.9 L78.6,146.8
 L78.7,144.8 L78.7,142.9 L78.8,140.9 L78.9,139.0 L79.0,137.0 L79.0,135.1 L79.1,133.3 L79.2,131.4
 L79.2,129.6 L79.3,127.8 L79.4,126.0 L79.4,124.3 L79.5,122.5 L79.6,120.8 L79.6,119.1 L79.7,117.5
 L79.8,115.8 L79.8,114.1 L79.9,112.5 L80.0,110.9 L80.0,109.3 L80.1,107.8 L80.2,106.2 L80.3,104.7
 L80.3,103.2 L80.4,101.7 L80.5,100.3 L80.5,98.8 L80.6,97.4 L80.7,96.1 L80.7,94.7 L80.8,93.4
 L80.9,92.1 L80.9,90.9 L81.0,89.6 L81.1,88.4 L81.1,87.2 L81.2,86.0 L81.3,84.9 L81.3,83.8
 L81.4,82.7 L81.5,81.6 L81.6,80.6 L81.6,79.5 L81.7,78.5 L81.8,77.5 L81.8,76.5 L81.9,75.5
 L82.0,74.6 L82.0,73.8 L82.1,72.9 L82.2,72.0 L82.2,71.2 L82.3,70.4 L82.4,69.7 L82.4,68.9
 L82.5,68.1 L82.6,67.5 L82.6,66.8 L82.7,66.2 L82.8,65.6 L82.9,65.0 L82.9,64.5 L83.0,63.9
 L83.1,63.4 L83.1,62.9 L83.2,62.5 L83.3,62.0 L83.3,61.6 L83.4,61.2 L83.5,60.8 L83.5,60.5
 L83.6,60.2 L83.7,59.9 L83.7,59.6 L83.8,59.3 L83.9,59.1 L84.0,58.9 L84.0,58.7 L84.1,58.6
 L84.2,58.4 L84.2,58.3 L84.3,58.3 L84.4,58.2 L84.5,58.3 L84.6,58.3 L84.7,58.4 L84.8,58.5
 L84.8,58.7 L84.9,58.8 L85.0,59.0 L85.0,59.2 L85.1,59.4 L85.2,59.6 L85.3,59.9 L85.3,60.2
 L85.4,60.5 L85.5,60.8 L85.5,61.2 L85.6,61.5 L85.7,62.0 L85.7,62.4 L85.8,62.9 L85.9,63.3
 L85.9,63.9 L86.0,64.4 L86.1,64.9 L86.1,65.5 L86.2,66.2 L86.3,66.8 L86.3,67.4 L86.4,68.1
 L86.5,68.8 L86.6,69.6 L86.6,70.3 L86.7,71.0 L86.8,71.8 L86.8,72.6 L86.9,73.5 L87.0,74.3
 L87.0,75.2 L87.1,76.1 L87.2,77.0 L87.2,77.9 L87.3,78.8 L87.4,79.8 L87.4,80.9 L87.5,81.9
 L87.6,82.9 L87.6,84.0 L87.7,85.1 L87.8,86.2 L87.9,87.3 L87.9,88.5 L88.0,89.7 L88.1,90.9
 L88.1,92.2 L88.2,93.4 L88.3,94.6 L88.3,95.9 L88.4,97.3 L88.5,98.6 L88.5,99.9 L88.6,101.2
 L88.7,102.6 L88.7,104.0 L88.8,105.4 L88.9,106

corrected reference rise height

In [27]:
riseHeight

19.160531485066464

In [28]:
NUnit.Framework.Assert.AreEqual(19.15, Fig15[0,0].dataGroups[0].Values.Last(), 5e-3, 
                $"Terminal rise height for Omega = 0.1 does not match reference value.");

In [29]:
NUnit.Framework.Assert.AreEqual(18.93, Fig15[0,1].dataGroups[0].Values.Last(), 5e-3, 
                $"Terminal rise height for Omega = 0.5 does not match reference value.");

In [30]:
NUnit.Framework.Assert.AreEqual(19.39, Fig15[1,0].dataGroups[0].Values.Last(), 5e-3, 
                $"Terminal rise height for Omega = 1 does not match reference value.");

In [31]:
NUnit.Framework.Assert.AreEqual(19.16, Fig15[1,1].dataGroups[0].Values.Last(), 5e-3, 
                $"Terminal rise height for Omega = 10 does not match reference value.");

In [32]:
NUnit.Framework.Assert.AreEqual(19.16, Fig15[1,1].dataGroups[1].Values.Last(), 5e-3, 
                $"Terminal rise height for Omega = 100 does not match reference value.");

## Check against origin data set

In [ ]:
public static void CheckAgainstOriginData(List<Plot2Ddata>[] StudyData, List<Plot2Ddata>[] OriginData, string omega, double errThrsh, bool output = false) {

    var Sgrp = StudyData[0].ElementAt(1).dataGroups.Where(grp => grp.Name.Contains(omega)).Single();

    //foreach (var Sgrp in StudyGrps) {

        var Ogrp = OriginData[0].ElementAt(1).dataGroups.Where(grp => grp.Name.Contains(omega)).Single();

        var errorData = ISessionInfoExtensions.ComputeErrorNormsForLogData(Sgrp, Ogrp);

        if(output) {
            var nameData = Sgrp.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
            Console.WriteLine($"checking {nameData[1]}:");  

            Console.WriteLine($"    l1-error norm = {errorData["l_1 error norm"]}");
            Console.WriteLine($"    l2-error norm = {errorData["l_2 error norm"]}");
            Console.WriteLine($"    linf-error norm = {errorData["l_inf error norm"]}");
        }

        NUnit.Framework.Assert.LessOrEqual(errorData["l_1 error norm"], errThrsh,  
            $"l_1 error norm larger than allowed: {errorData["l_1 error norm"]}");

        NUnit.Framework.Assert.LessOrEqual(errorData["l_2 error norm"], errThrsh,  
            $"l_2 error norm larger than allowed: {errorData["l_2 error norm"]}");

        NUnit.Framework.Assert.LessOrEqual(errorData["l_inf error norm"], errThrsh,  
            $"l_inf error norm larger than allowed: {errorData["l_inf error norm"]}");

    //}
}

In [ ]:
//CheckAgainstOriginData(cleanEvalData, cleanEvalDataOrigin, "_Omega05_", 0.9, true);

checking Omega05:
    l1-error norm = 0
    l2-error norm = 0
    linf-error norm = 0


In [35]:
CheckAgainstOriginData(cleanEvalData, cleanEvalDataOrigin, "_Omega1_", 0.05, true);

checking Omega1:
    l1-error norm = 0.02401688738722856
    l2-error norm = 0.02935082497514628
    linf-error norm = 0.04942397813323705


In [36]:
CheckAgainstOriginData(cleanEvalData, cleanEvalDataOrigin, "_Omega10_", 0.005, true);

checking Omega10:
    l1-error norm = 0.0009250069975988715
    l2-error norm = 0.0010213145311949241
    linf-error norm = 0.002214049639840168


In [ ]:
CheckAgainstOriginData(cleanEvalData, cleanEvalDataOrigin, "_Omega100_", 0.04, true);

checking Omega100:
    l1-error norm = 0
    l2-error norm = 0
    linf-error norm = 0
